<a href="https://colab.research.google.com/github/Amonstarr/Twitter-Classification-Using-Indobert/blob/main/Indobert_engine_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets evaluate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

In [ ]:
NUM_LABELS =  8#@param {type:"integer"}
#TRAIN_PATH_DATA_INPUT = "/content/drive/MyDrive/Satria Data/dataset_penyisihan_bdc_2024_clean.csv" #@param {type:"string"}
TRAIN_PATH_DATA_INPUT = "/content/drive/MyDrive/Satria Data/Balance/clean_train_balanced.csv" #@param {type:"string"}
TEST_PATH_DATA_INPUT = "/content/drive/MyDrive/Satria Data/datatest_penyisihan_bdc_2024_clean.csv" #@param {type:"string"}

In [ ]:
df_train = pd.read_csv(TRAIN_PATH_DATA_INPUT)


In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
import pandas as pd
# Training Data Handling
df = pd.read_csv(TRAIN_PATH_DATA_INPUT)

from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(df, shuffle=True, test_size=0.2)

train = train_df.drop(columns="raw", axis=1)
train = train_df.rename(columns={
    "processed" : "text",
    "labels" : "label"
})

eval = eval_df.drop(columns="raw", axis=1)
eval = eval_df.rename(columns={
    "processed" : "text",
    "labels" : "label"
})
######### Ini Harusnya ada di Pre-Processing ############
# mapping = {
#     -1 : 0,
#     1 : 1,
# }
# df['label']= df['label'].map(mapping)
# df = df.reset_index(drop=True)
#########################################################



# Testing Data Handling
# df_test = pd.read_csv(TEST_PATH_DATA_INPUT)
# df_test = df_test.drop(columns="raw", axis=1)
# df_test = df_test.rename(columns={
#     "processed" : "text",
#     "labels" : "label"
# })

In [ ]:
train.to_csv("./train.csv", index=False)
eval.to_csv("./eval.csv", index=False)
# df_test.to_csv("./test.csv", index=False)

In [ ]:
from datasets import load_dataset
files = {"train": "/content/train.csv",
         "eval": "/content/eval.csv",
        #  "test":"/content/test.csv"
        }
dataset = load_dataset('csv', data_files=files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2", use_fast=True)
tokenizer.add_tokens([ 'NaN' ])

1

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding='max_length', max_length=256, truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4174 [00:00<?, ? examples/s]

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

In [ ]:
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["eval"]

# Modelling


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p2", num_labels=NUM_LABELS)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

# training_args = TrainingArguments("test_trainer")

#training_args = TrainingArguments(
#    output_dir="./running_trainer",
#    do_train=True,
#    eval_strategy="steps",
#    learning_rate=5e-8,
#    num_train_epochs=3,
#    save_strategy="epoch",
#    save_steps=200,
#)


In [ ]:
training_args = TrainingArguments(
    output_dir="./model_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
)


In [ ]:
from transformers import Trainer
import numpy as np
import evaluate
# metric = evaluate.load("accuracy")

# #metric = load_metric("accuracy")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return metric.compute(predictions=predictions, references=labels)

#atau pakai ini
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)
training_history = trainer.train()
evaluation_history = trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.936800,0.782567,0.720710
2,0.168602,1.064073,0.768199,0.710238
3,0.168602,1.127732,0.775862,0.716996


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [ ]:
evaluation_history

{'eval_loss': 0.9373748302459717,
 'eval_accuracy': 0.7825670498084292,
 'eval_f1_macro': 0.7207098514969603,
 'eval_runtime': 14.5684,
 'eval_samples_per_second': 71.662,
 'eval_steps_per_second': 4.53,
 'epoch': 3.0}

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/Satria Data/Model_IndoBret_kastemargs"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }